In [2]:
# Import data

from datasets import load_dataset
ds = load_dataset("ucirvine/sms_spam")

In [3]:
import pandas as pd

df = pd.DataFrame(ds['train'])
print(df.head())

                                                 sms  label
0  Go until jurong point, crazy.. Available only ...      0
1                    Ok lar... Joking wif u oni...\n      0
2  Free entry in 2 a wkly comp to win FA Cup fina...      1
3  U dun say so early hor... U c already then say...      0
4  Nah I don't think he goes to usf, he lives aro...      0


In [4]:
# convert training dataframe to text file

df.to_csv('training_data.txt', sep='\t', index=False)

In [7]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Training data frame:", train_df.head())
print("Training set size:", len(train_df))
print("Testing data frame:", test_df.head())
print("Testing set size:", len(test_df))

Training data frame:                                                     sms  label
1350  FREE2DAY sexy St George's Day pic of Jordan!Tx...      1
5544         Armand says get your ass over to epsilon\n      0
1168          Lol now I'm after that hot air balloon!\n      0
5551  You know, wot people wear. T shirts, jumpers, ...      0
5320  Good morning, my Love ... I go to sleep now an...      0
Training set size: 4459
Testing data frame:                                                     sms  label
3690                        You still coming tonight?\n      0
3527  "HEY BABE! FAR 2 SPUN-OUT 2 SPK AT DA MO... DE...      0
724          Ya even those cookies have jelly on them\n      0
3370  Sorry i've not gone to that place. I.ll do so ...      0
468             When are you going to ride your bike?\n      0
Testing set size: 1115


In [9]:
# Break all words into a list of small words and convert to lower case and remove punctuation

train_df['words'] = train_df['sms'].str.lower()
train_df['words'] = train_df['words'].str.split()
train_df['words'] = train_df['words'].apply(lambda x: [word.strip('.,!?*%^&"()0123456789|\\//#@''<>~`:;') for word in x])

print(train_df.words)

1350    [free2day, sexy, st, george's, day, pic, of, j...
5544    [armand, says, get, your, ass, over, to, epsilon]
1168      [lol, now, i'm, after, that, hot, air, balloon]
5551    [you, know, wot, people, wear, t, shirts, jump...
5320    [good, morning, my, love, , i, go, to, sleep, ...
                              ...                        
3772    [hi, wlcome, back, did, wonder, if, you, got, ...
5191    [ree, entry, in, , a, weekly, comp, for, a, ch...
5226    [oh, fuck, juswoke, up, in, a, bed, on, a, boa...
5390      [not, much, no, fights, it, was, a, good, nite]
860      [did, he, just, say, somebody, is, named, tampa]
Name: words, Length: 4459, dtype: object


In [18]:
# define spam and ham for training dataframes
df_spam = train_df[train_df['label'] == 0]
df_ham = train_df[train_df['label'] == 1]
print(df_spam.head())
print(df_ham.head())

                                                    sms  label  \
5544         Armand says get your ass over to epsilon\n      0   
1168          Lol now I'm after that hot air balloon!\n      0   
5551  You know, wot people wear. T shirts, jumpers, ...      0   
5320  Good morning, my Love ... I go to sleep now an...      0   
4323  Sorry . I will be able to get to you. See you ...      0   

                                                  words  
5544  [armand, says, get, your, ass, over, to, epsilon]  
1168    [lol, now, i'm, after, that, hot, air, balloon]  
5551  [you, know, wot, people, wear, t, shirts, jump...  
5320  [good, morning, my, love, , i, go, to, sleep, ...  
4323  [sorry, , i, will, be, able, to, get, to, you,...  
                                                    sms  label  \
1350  FREE2DAY sexy St George's Day pic of Jordan!Tx...      1   
1741  UR GOING 2 BAHAMAS! CallFREEFONE 08081560665 a...      1   
3954  Refused a loan? Secured or Unsecured? Can't ge...  

In [19]:
# write each word in the list of words to consist 
words_counts = train_df['words'].explode().value_counts()
words_counts_spam = df_spam['words'].explode().value_counts()
words_counts_ham = df_ham['words'].explode().value_counts()

print(words_counts)
print(words_counts_spam)
print(words_counts_ham)


# Create a dictionary of words and their counts for spam and ham
words_dic = {}
for word in words_counts.index:
    words_dic[word] = {
        'spam': words_counts_spam.get(word, 0),
        'ham': words_counts_ham.get(word, 0)
    }   

print(words_dic)


words
           2662
i          1804
to         1785
you        1711
a          1149
           ... 
juswoke       1
boatin        1
docks         1
spinout       1
fights        1
Name: count, Length: 7947, dtype: int64
words
i          1756
you        1485
           1395
to         1247
the         901
           ... 
boatin        1
docks         1
spinout       1
gossip        1
fights        1
Name: count, Length: 6764, dtype: int64
words
              1267
to             538
a              306
call           269
you            226
              ... 
now!reply        1
june             1
soundtrack       1
stdtxtrate       1
ree              1
Name: count, Length: 2105, dtype: int64
{'': {'spam': np.int64(1395), 'ham': np.int64(1267)}, 'i': {'spam': np.int64(1756), 'ham': np.int64(48)}, 'to': {'spam': np.int64(1247), 'ham': np.int64(538)}, 'you': {'spam': np.int64(1485), 'ham': np.int64(226)}, 'a': {'spam': np.int64(843), 'ham': np.int64(306)}, 'the': {'spam': np.int64(901), 'ha

In [20]:
# Convert the dictionary to a dataframe
words_df = pd.DataFrame.from_dict(words_dic, orient='index')
print(words_df.head())

     spam   ham
     1395  1267
i    1756    48
to   1247   538
you  1485   226
a     843   306


In [21]:
# Calculate the total number of spam and ham messages
total_spam = len(df_spam)
total_ham = len(df_ham)

print(f'Total spam messages: {total_spam}')
print(f'Total ham messages: {total_ham}')

Total spam messages: 3873
Total ham messages: 586


In [22]:
# Calculate the probability of each word being spam or ham i.e. P(word|spam) and P(word|ham)
words_df['p_word_spam'] = words_df['spam'] / total_spam
words_df['p_word_ham'] = words_df['ham'] / total_ham

print(words_df.head())

     spam   ham  p_word_spam  p_word_ham
     1395  1267     0.360186    2.162116
i    1756    48     0.453395    0.081911
to   1247   538     0.321973    0.918089
you  1485   226     0.383424    0.385666
a     843   306     0.217661    0.522184


In [23]:
# Calculate the probability of massaging being spam or ham i.e. P(spam) and P(ham)
p_spam = total_spam / len(train_df)
p_ham = total_ham / len(train_df)
print(f'P(spam): {p_spam}')
print(f'P(ham): {p_ham}')

P(spam): 0.8685803991926441
P(ham): 0.1314196008073559


In [27]:
# Calculate P(spam|word) and P(ham|word) using Bayes theorem
import numpy as np
def spam_probability(word, message):
    if word in words_df.index:
        p_word_spam = words_df.loc[word, 'p_word_spam']
        p_word_ham = words_df.loc[word, 'p_word_ham']
    else:
        p_word_spam = 0
        p_word_ham = 0

    p_spam_given_word = (p_word_spam * p_spam) / ((p_word_spam * p_spam) + (p_word_ham * p_ham))
    p_ham_given_word = (p_word_ham * p_ham) / ((p_word_spam * p_spam) + (p_word_ham * p_ham))

    return p_spam_given_word, p_ham_given_word

print(spam_probability('free', 'You have won a free ticket!'))

(np.float64(0.23333333333333336), np.float64(0.7666666666666667))


In [50]:
# Create Class of Naive Bayes Classifier

class NaiveBayes_Classifier:
    def __init__(self):
        self.p_spam = 0
        self.p_ham = 0
        self.words_df = None

    def train(self, train_df):
        self.train_df = train_df
        self.df_spam = train_df[train_df['label'] == 0]
        self.df_ham = train_df[train_df['label'] == 1]

        self.total_spam = len(self.df_spam)
        self.total_ham = len(self.df_ham)

        self.p_spam = self.total_spam / len(self.train_df)
        self.p_ham = self.total_ham / len(self.train_df)

        words_counts = self.train_df['words'].explode().value_counts()
        words_counts_spam = self.df_spam['words'].explode().value_counts()
        words_counts_ham = self.df_ham['words'].explode().value_counts()

        words_dic = {}
        for word in words_counts.index:
            words_dic[word] = {
                'spam': words_counts_spam.get(word, 0),
                'ham': words_counts_ham.get(word, 0)
            }   

        self.words_df = pd.DataFrame.from_dict(words_dic, orient='index')
        self.words_df['p_word_spam'] = self.words_df['spam'] / self.total_spam
        self.words_df['p_word_ham'] = self.words_df['ham'] / self.total_ham
    
    def spam_probability(self, word):
        if word not in self.words_df.index:
            return 1.0, 1.0  # Neutral contribution for unseen words
        
        p_word_spam = self.words_df.loc[word, 'p_word_spam']
        p_word_ham = self.words_df.loc[word, 'p_word_ham']
        denominator = p_word_spam * self.p_spam + p_word_ham * self.p_ham
        
        if denominator == 0:
            return 1.0, 1.0
        
        return (
                p_word_spam * self.p_spam / denominator,
                p_word_ham * self.p_ham / denominator
            )

    def predict(self, message):
        words = message.lower().split()
        words = [word.strip('.,!?*%^&"()0123456789|\\//#@''<>~`:;') for word in words]
        p_spam_given_message = self.p_spam
        p_ham_given_message = self.p_ham

        for word in words:
            p_word_spam, p_word_ham = self.spam_probability(word)
            p_spam_given_message *= p_word_spam
            p_ham_given_message *= p_word_ham

        if p_spam_given_message > p_ham_given_message:
            return 0  # spam
        else:
            return 1  # ham
            

In [51]:
# Fit the model
model = NaiveBayes_Classifier()
model.train(train_df)

In [52]:
# Predict the model
predict_train_df = train_df['sms'].apply(model.predict)

In [53]:
# Find Accuracy of Train data
from sklearn.metrics import accuracy_score, precision_score

accuracy_train = accuracy_score(train_df['label'], predict_train_df)
print(accuracy_train)

0.9970845481049563


In [54]:
# Test the testing data dataframe
predict_test_df = test_df['sms'].apply(model.predict)
accuracy_test = accuracy_score(test_df['label'], predict_test_df)
print(accuracy_test)

0.9443946188340807
